**by: [João Ataíde](https://www.joaoataide.com)**
# **CAP 06: Estatística Inferencial**

---


In [1]:
#importar bibliotecas
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt
import seaborn as sns
import statistics as st
import scipy.stats as stats
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency, spearmanr

## **Amostragem**

* `População:` conjunto de todos os indivíduos, objetos ou elementos a serem estudados. O `censo` é o estudo de todos os elementos da população. Uma população pode ser `finita`, quando permite contagem (ex.: quantidade de empregados de uma empresa), ou `infinita`, quando não permite (ex.: número de produtos de uma fábrica ao longo do tempo).

* `Amostra:` subconjunto extraído da população, a partir do qual usamos procedimentos estatísticos apropriados para generalizar, inferir ou tirar conclusões sobre a população.

### **Tipos de amostragem**
- `Aleatória ou probabilística:` todos os elementos da população têm probabilidade conhecida (e diferente de zero) de serem selecionados.
    * `Simples`
    * `Sistemática`
    * `Estratificada`
    * `Por conglomerados`

    * Vantagens: seleção rigorosa, sem subjetividade, e possibilidade de determinar matematicamente o tamanho da amostra e o grau de confiança.

    * Desvantagens: dificuldade em obter a lista completa da população, e a seleção aleatória pode gerar uma amostra muito dispersa geograficamente.

- `Não aleatória ou não probabilística:` a probabilidade de seleção de alguns ou de todos os elementos da população é desconhecida.
    * `Por conveniência`
    * `Por julgamento`
    * `Por cotas`
    * `Bola de neve`

    * Vantagens: menor custo, menor tempo de estudo e menos mão de obra.

    * Desvantagens: há unidades da população sem chance de serem escolhidas, pode haver viés e não é possível saber o grau de confiança dos resultados.

### **Amostragem Aleatória**

#### **Amostragem Aleatória Simples**

Método mais simples e um dos mais importantes de seleção de amostras.

Planejamento:
1. Usar um procedimento aleatório para sortear elementos da população, todos com a mesma probabilidade.
2. Repetir o processo até obter uma amostra com $n$ observações.
3. O elemento sorteado pode ser retirado da população (`sem reposição`) ou devolvido (`com reposição`), permitindo neste caso que uma unidade seja sorteada mais de uma vez.

In [2]:
np.random.seed(42)
random.seed(42)

populacao = {
    'ID': list(range(1, 31)),
    'Valor': [6.4, 6.2, 7.0, 6.8, 7.2, 6.4, 6.5, 7.1, 6.8, 6.9, 7.0, 7.1, 6.6, 6.8, 6.7,
              6.3, 6.6, 7.2, 7.0, 6.9, 6.8, 6.7, 6.5, 7.2, 6.8, 6.9, 7.0, 6.7, 6.9, 6.9]
}


df_populacao = pd.DataFrame(populacao).set_index("ID")
print(df_populacao.shape)
df_populacao.head()

(30, 1)


,Valor
ID,
1,6.4
2,6.2
3,7.0
4,6.8
5,7.2


##### *Sem Reposição*

In [3]:
# Amostragem aleatória simples sem reposição do DataFrame
amostra_sem_reposicao = df_populacao.sample(n=10)
print(amostra_sem_reposicao.shape)
amostra_sem_reposicao

(10, 1)


,Valor
ID,
28,6.7
16,6.3
24,7.2
18,7.2
9,6.8
10,6.9
29,6.9
25,6.8
13,6.6


##### *Com Reposição*

In [4]:
# Amostragem aleatória simples com reposição do DataFrame
amostra_com_reposicao = df_populacao.sample(n=10, replace=True)
print(amostra_com_reposicao.shape)
amostra_com_reposicao

(10, 1)


,Valor
ID,
10,6.9
28,6.7
28,6.7
16,6.3
15,6.7
30,6.9
30,6.9
15,6.7
30,6.9


#### **Amostragem Aleatória Sistemática**

Basicamente, na amostragem sistemática os elementos estão ordenados e são retirados periodicamente. *Ex: retirar uma peça a cada 50 de uma linha de produção.*

Planejamento:
1. Determinar o intervalo de seleção
$k=N/n$
2. Sortear o ponto de partida entre os $k$ primeiros elementos, para introduzir aleatoriedade.
3. A partir dele, selecionar um elemento a cada $k$ até completar a amostra de tamanho $n$.

In [5]:
# Amostragem sistemática a cada 5 elementos
k = 5
inicio = random.randint(0, k - 1)  # Ponto de partida sorteado
indices_amostra = list(range(inicio, len(df_populacao), k))
amostra_sistematica = df_populacao.iloc[indices_amostra]
amostra_sistematica

,Valor
ID,
1,6.4
6,6.4
11,7.0
16,6.3
21,6.8
26,6.9


#### **Amostragem Aleatória Estratificada**

Esse tipo divide uma população heterogênea em subpopulações (estratos) homogêneas e retira uma amostra aleatória de cada estrato. Na amostragem `uniforme` sorteamos o mesmo número de elementos em cada estrato (recomendada quando os estratos têm tamanhos próximos). Na `proporcional` o número de elementos sorteados em cada estrato é proporcional ao seu tamanho na população.

In [6]:
# Definir as classes de valor
classes_valor = {
    'A': (5., 6.5),
    'B': (6.5, 7.),
    'C': (7., 7.5)
}

In [7]:
for classe, (lim_inferior, lim_superior) in classes_valor.items():
    df_populacao.loc[(df_populacao['Valor'] > lim_inferior) & (df_populacao['Valor'] <= lim_superior), 'Classe'] = classe

df_populacao.head()

,Valor,Classe
ID,,
1,6.4,A
2,6.2,A
3,7.0,B
4,6.8,B
5,7.2,C


##### *Uniforme*

In [8]:
n = 2  # ou qualquer outro número que seja adequado para o seu caso
amostra_uniforme = df_populacao.groupby('Classe').apply(lambda x: x.sample(n))

# Resetar o índice
amostra_uniforme.reset_index(drop=True, inplace=True)
amostra_uniforme

C:\Users\jv_at\AppData\Local\Temp\ipykernel_83988\1722400454.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  amostra_uniforme = df_populacao.groupby('Classe').apply(lambda x: x.sample(n))


,Valor,Classe
0,6.2,A
1,6.4,A
2,7.0,B
3,6.6,B
4,7.1,C
5,7.2,C


##### *Proporcional*

In [9]:
tamanho_amostra = 0.3

# Usamos train_test_split para estratificar com base na classe
_, amostra_proporcional = train_test_split(df_populacao,
                                           stratify=df_populacao['Classe'],
                                           test_size=tamanho_amostra,
                                           random_state=42)

amostra_proporcional

,Valor,Classe
ID,,
10,6.9,B
20,6.9,B
14,6.8,B
15,6.7,B
7,6.5,A
29,6.9,B
18,7.2,C
16,6.3,A
11,7.0,B


#### **Amostragem Aleatória por Conglomerados**

Na amostragem por conglomerados a população é dividida em grupos de unidades elementares (conglomerados), e o sorteio é feito sobre os grupos, não sobre os indivíduos.

Planejamento:
1. Dividir a população em $M$ conglomerados, de tamanhos não necessariamente iguais.
2. Sortear uma amostra de $m < M$ conglomerados.
3. Em `1 estágio`, todos os elementos dos conglomerados sorteados entram na amostra. Em `2 estágios`, sorteia-se ainda uma amostra de elementos dentro de cada conglomerado sorteado.

In [10]:
# Definir o número de conglomerados
num_conglomerados = 5

In [11]:
# Definir o tamanho do conglomerado
tamanho_conglomerado = len(df_populacao) // num_conglomerados

# Atribuir o conglomerado primário
conglomerados = list(range(1, num_conglomerados + 1))
random.shuffle(conglomerados)
conglomerados *= len(df_populacao) // num_conglomerados + 1
df_populacao['ConglomeradoPrimario'] = conglomerados[:len(df_populacao)]

# Atribuir o conglomerado secundário
conglomerados = list(range(1, num_conglomerados + 1))
random.shuffle(conglomerados)
conglomerados *= len(df_populacao) // num_conglomerados + 1
df_populacao['ConglomeradoSecundario'] = conglomerados[:len(df_populacao)]

In [12]:
print(df_populacao.shape)
df_populacao.head()

(30, 4)


,Valor,Classe,ConglomeradoPrimario,ConglomeradoSecundario
ID,,,,
1,6.4,A,2,5
2,6.2,A,4,4
3,7.0,B,5,3
4,6.8,B,3,1
5,7.2,C,1,2


##### *1 Estágio*

In [13]:
# Conglomerados primários
conglomerados_primarios = df_populacao['ConglomeradoPrimario'].unique()

# Definir o número de conglomerados primários a serem selecionados
num_conglomerados_primarios_amostra = 2

# Selecionar aleatoriamente os conglomerados primários
conglomerados_primarios_amostra = random.sample(list(conglomerados_primarios), k=num_conglomerados_primarios_amostra)

# Realizar a amostragem por conglomerado em um estágio
amostra_um_estagio = df_populacao[df_populacao['ConglomeradoPrimario'].isin(conglomerados_primarios_amostra)]
print(amostra_um_estagio.shape)
amostra_um_estagio

(12, 4)


,Valor,Classe,ConglomeradoPrimario,ConglomeradoSecundario
ID,,,,
4,6.8,B,3,1
5,7.2,C,1,2
9,6.8,B,3,1
10,6.9,B,1,2
14,6.8,B,3,1
15,6.7,B,1,2
19,7.0,B,3,1
20,6.9,B,1,2
24,7.2,C,3,1


##### *2 Estágio*

In [14]:
# Conglomerados secundários
conglomerados_secundarios = amostra_um_estagio['ConglomeradoSecundario'].unique()

# Definir o número de conglomerados secundários a serem selecionados
num_conglomerados_secundarios_amostra = 1

# Selecionar aleatoriamente os conglomerados secundários
conglomerados_secundarios_amostra = random.sample(list(conglomerados_secundarios), k=num_conglomerados_secundarios_amostra)

# Realizar a amostragem por conglomerado em dois estágios
amostra_dois_estagios = amostra_um_estagio[amostra_um_estagio['ConglomeradoSecundario'].isin(conglomerados_secundarios_amostra)]
print(amostra_dois_estagios.shape)
amostra_dois_estagios

(6, 4)


,Valor,Classe,ConglomeradoPrimario,ConglomeradoSecundario
ID,,,,
4,6.8,B,3,1
9,6.8,B,3,1
14,6.8,B,3,1
19,7.0,B,3,1
24,7.2,C,3,1
29,6.9,B,3,1


### **Amostragem Não Aleatória**

#### **Amostragem por Conveniência**

É empregada quando a participação é voluntária ou quando os elementos são escolhidos por conveniência ou simplicidade, como amigos, vizinhos e estudantes.

In [15]:
ids_conveniencia = [5, 10, 15, 20, 25]
amostra_conveniencia = df_populacao[df_populacao.index.isin(ids_conveniencia)]
amostra_conveniencia

,Valor,Classe,ConglomeradoPrimario,ConglomeradoSecundario
ID,,,,
5,7.2,C,1,2
10,6.9,B,1,2
15,6.7,B,1,2
20,6.9,B,1,2
25,6.8,B,1,2


#### **Amostragem por Julgamento ou Intencional**

A amostra é escolhida de forma intencional, a partir do julgamento de um especialista ou de uma regra de negócio, o que pode levar a conclusões equivocadas e enviesar a análise. *O emprego dessa técnica requer bom conhecimento da população e dos elementos selecionados.*

*Ex: uma pequena empresa escolhe entrevistar um cliente específico porque ele se encaixa no perfil de cliente que ela quer fortalecer.*

In [16]:
amostra_julgamento = df_populacao[(df_populacao['Valor'] >= 6) & (df_populacao['Valor'] <= 6.5)]
amostra_julgamento

,Valor,Classe,ConglomeradoPrimario,ConglomeradoSecundario
ID,,,,
1,6.4,A,2,5
2,6.2,A,4,4
6,6.4,A,2,5
7,6.5,A,4,4
16,6.3,A,2,5
23,6.5,A,5,3


#### **Amostragem por Cotas**

Apresenta maior rigor que as demais amostragens não aleatórias e é muito usada em pesquisas de mercado e de opinião eleitoral. Pode ser vista como uma amostragem estratificada não probabilística.

Procedimento:
1. Selecionar as variáveis de controle, ou seja, as características relevantes da população.
2. Determinar a proporção da população em cada categoria dessas variáveis.
3. Dimensionar as cotas (número de pessoas a serem entrevistadas em cada categoria) de modo que as proporções da amostra sejam iguais às da população.

In [17]:
# Calcular o tamanho da amostra para cada classe
n = 1

# Amostra por quota
amostra_por_quota = df_populacao.groupby('Classe').apply(lambda x: x.sample(min(len(x), n)))

# Resetar o índice
amostra_por_quota.reset_index(drop=True, inplace=True)
amostra_por_quota

C:\Users\jv_at\AppData\Local\Temp\ipykernel_83988\1503506167.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  amostra_por_quota = df_populacao.groupby('Classe').apply(lambda x: x.sample(min(len(x), n)))


,Valor,Classe,ConglomeradoPrimario,ConglomeradoSecundario
0,6.3,A,2,5
1,7.0,B,2,5
2,7.1,C,4,4


#### **Amostragem Bola de Neve ou Propagação Geométrica**
Muito usada quando a população é rara, de difícil acesso ou até desconhecida, por exemplo uma empresa de recrutamento que precisa de profissionais com um perfil específico. É comum em pesquisas sociais. Basicamente identificamos um ou mais indivíduos da população-alvo, que indicam outros, repetindo o processo até atingir o tamanho desejado.

In [18]:
# Adicionando três elementos raros à população
df_populacao.loc[31] = [10, "Z", 0, 0]
df_populacao.loc[32] = [8, "Z", 0, 0]
df_populacao.loc[33] = [9.3, "Z", 0, 0]

In [19]:
def snowball_sampling(df, initial_sample_size, iterations, classe):
    # Selecione uma amostra inicial aleatória
    initial_sample = df.sample(initial_sample_size)

    for i in range(iterations):
        # Encontre os valores únicos na coluna 'Classe' do nosso conjunto de amostras atual
        unique_values = initial_sample[classe].unique()

        # Selecionar elementos específicos (Passar Regra de negócio ou análise)
        new_samples = df[df[classe] > 7]

        # Adicione as novas amostras ao conjunto de amostras inicial
        initial_sample = pd.concat([initial_sample, new_samples])

        # Remova duplicatas
        initial_sample = initial_sample.drop_duplicates()

    return initial_sample

In [20]:
sample = snowball_sampling(df_populacao, initial_sample_size=5, iterations=100, classe ="Valor")
sample

,Valor,Classe,ConglomeradoPrimario,ConglomeradoSecundario
ID,,,,
19,7.0,B,3,1
10,6.9,B,1,2
14,6.8,B,3,1
6,6.4,A,2,5
21,6.8,B,2,5
5,7.2,C,1,2
8,7.1,C,5,3
12,7.1,C,4,4
18,7.2,C,5,3


## **Tamanho da Amostra**

**Fatores que determinam o tamanho da amostra:**

1. Características da população, como a variância $σ^2$ e o tamanho $N$.
2. Distribuição amostral do estimador.
3. Precisão e confiança exigidas, especificando o erro de estimação $B$ (máxima diferença que o pesquisador admite entre o parâmetro populacional e a estimativa).
4. Custo: quanto maior a amostra, maior o custo da operação.
5. Custo vs. erro: amostras maiores reduzem o erro amostral, mas aumentam o custo, e é preciso equilibrar os dois.
6. Técnicas estatísticas que serão usadas, pois algumas exigem amostras maiores que outras.

#### **Tamanho da amostra aleatória simples**

Cálculo do tamanho da amostra para estimar a média e a proporção, em populações finitas e infinitas, com nível de confiança $1 - \alpha$.

`Média`

Com erro de estimação $B$, média populacional $\mu$ e média amostral $\overline X$, queremos $|\mu - \overline X| \leq B$:

* População infinita: $n = \frac{z^2_{\alpha/2}\cdot\sigma^2}{B^2}$
* População finita: $n = \frac{N\cdot z^2_{\alpha/2}\cdot\sigma^2}{(N-1)\cdot B^2 + z^2_{\alpha/2}\cdot\sigma^2}$

`Proporção`

Com erro $B$, proporção populacional $p$ e proporção amostral $\hat p$, queremos $|p - \hat p| \leq B$:

* População infinita: $n = \frac{z^2_{\alpha/2}\cdot p(1-p)}{B^2}$
* População finita: $n = \frac{N\cdot z^2_{\alpha/2}\cdot p(1-p)}{(N-1)\cdot B^2 + z^2_{\alpha/2}\cdot p(1-p)}$

`OBS:` *Quando $p$ é desconhecido usa-se $p = 0.5$, que leva ao maior tamanho de amostra.*

In [21]:
random.seed(42)

dados = {'Valor': [],
         'ValorBinario':[]}

for _ in range(10000): # Gerando uma população de 10000 elementos
    valor = random.uniform(3.0, 10.0)  # Gerar valores aleatórios entre 3.0 e 10.0
    dados['Valor'].append(valor)

    valorb = random.randint(0, 1)
    dados['ValorBinario'].append(valorb)

df = pd.DataFrame(dados)

In [22]:
def tamanho_amostra_media(populacao, erro, confianca=0.95, finita=False):

    z = norm.ppf(1 - (1 - confianca) / 2)  # Valor crítico para o nível de confiança
    desvio_padrao = populacao.std()  # Desvio padrão da população
    N = populacao.size

    if finita:
        tamanho = (N * (z**2) * (desvio_padrao**2)) / ((N - 1) * (erro**2) + (z**2) * (desvio_padrao**2))
    else:
        tamanho = ((z**2) * (desvio_padrao**2)) / (erro**2)

    return math.ceil(tamanho)

def tamanho_amostra_proporcao(populacao, proporcao, erro, confianca=0.95, finita=False):

    z = norm.ppf(1 - (1 - confianca) / 2)  # Valor crítico para o nível de confiança
    N = populacao.size

    if finita:
        tamanho = (N * (z**2) * proporcao * (1 - proporcao)) / ((N - 1) * (erro**2) + (z**2) * proporcao * (1 - proporcao))
    else:
        tamanho = ((z**2) * proporcao * (1 - proporcao)) / (erro**2)

    return math.ceil(tamanho)

In [23]:
erro = 0.05
proporcao = 0.5

# Para uma população finita
tamanho_amostra_media_finita = tamanho_amostra_media(df["Valor"], erro, finita=True)
tamanho_amostra_proporcao_finita = tamanho_amostra_proporcao(df["Valor"], proporcao, erro, finita=True)


print(f"Tamanho da amostra MÉDIA (população finita): {tamanho_amostra_media_finita}")
print(f"Tamanho da amostra PROPORÇÃO (população finita): {tamanho_amostra_proporcao_finita}")

Tamanho da amostra MÉDIA (população finita): 3876
Tamanho da amostra PROPORÇÃO (população finita): 370


In [24]:
erro = 0.05
proporcao = 0.5

# Para uma população infinita
tamanho_amostra_media_infinita = tamanho_amostra_media(df["Valor"], erro)
tamanho_amostra_proporcao_infinita = tamanho_amostra_proporcao(df["Valor"], proporcao, erro)

print(f"Tamanho da amostra MÉDIA (população infinita): {tamanho_amostra_media_infinita}")
print(f"Tamanho da amostra PROPORÇÃO (população infinita): {tamanho_amostra_proporcao_infinita}")

Tamanho da amostra MÉDIA (população infinita): 6328
Tamanho da amostra PROPORÇÃO (população infinita): 385


#### **Tamanho da amostra sistemática**

A amostragem sistemática usa as mesmas expressões da amostragem aleatória simples, escolhendo a da média (variável quantitativa) ou a da proporção (variável binária) e a versão para população finita ou infinita.

#### **Tamanho da amostra estratificada**

Neste caso o cálculo considera a variância (ou a proporção) de cada estrato: usamos a expressão da média quando a variável é quantitativa e a da proporção quando é binária, seguindo a mesma lógica de populações finitas e infinitas.

In [25]:
random.seed(42)

# Cria o DataFrame
dados = {'Área': [], 'Value': [], 'Classe': []}

for _ in range(5285):
    dados['Área'].append('Exatas')
    dados['Value'].append(random.uniform(3.0, 50.0))
    dados['Classe'].append(1)

for _ in range(3877):
    dados['Área'].append('Humanas')
    dados['Value'].append(random.uniform(3.0, 50.0))
    dados['Classe'].append(1)

for _ in range(27724):
    dados['Área'].append('Biológicas')
    dados['Value'].append(random.uniform(3.0, 50.0))
    dados['Classe'].append(0)

df = pd.DataFrame(dados)

In [26]:
def tamanho_amostra_media_estratificada(df, estrato_col, quant_col, erro, confianca=0.95, finita=False):

    z = norm.ppf(1 - (1 - confianca) / 2)  # Valor crítico para o nível de confiança
    tamanho_total = 0

    for grupo in df[estrato_col].unique():
        populacao_grupo = df[df[estrato_col] == grupo]
        desvio_padrao = populacao_grupo[quant_col].std()  # Desvio padrão do estrato
        N = len(populacao_grupo)

        if finita:
            tamanho_grupo = (N * (z**2) * (desvio_padrao**2)) / ((N - 1) * (erro**2) + (z**2) * (desvio_padrao**2))
        else:
            tamanho_grupo = ((z**2) * (desvio_padrao**2)) / (erro**2)

        tamanho_total += math.ceil(tamanho_grupo)

    return tamanho_total


def tamanho_amostra_proporcao_estratificada(dados, proporcoes, tamanho_amostra_total, finita=False):

    # Peso de cada estrato: proporção esperada vezes o tamanho do estrato
    pesos = {classe: proporcao * len(dados[dados['Classe'] == classe]) for classe, proporcao in proporcoes.items()}
    soma_pesos = sum(pesos.values())

    # Distribui a amostra total entre os estratos de acordo com os pesos
    tamanho_amostras = {}
    for classe, peso in pesos.items():
        tamanho_classe = len(dados[dados['Classe'] == classe])
        tamanho_estrato = (peso / soma_pesos) * tamanho_amostra_total

        if finita:
            tamanho_estrato = tamanho_estrato / (1 + (tamanho_estrato / tamanho_classe))

        tamanho_amostras[classe] = round(tamanho_estrato)

    return tamanho_amostras

In [27]:
erro = 1.0  # Erro máximo de 1 unidade na média
proporcoes = {0: 0.5, 1: 0.5}

# Para uma população finita
tamanho_amostra_media_finita = tamanho_amostra_media_estratificada(df, 'Classe', 'Value', erro, finita=True)
tamanho_amostra_proporcao_finita = tamanho_amostra_proporcao_estratificada(df, proporcoes, tamanho_amostra_media_finita, finita=True)

print(f"Tamanho da amostra MÉDIA (população finita): {tamanho_amostra_media_finita}")
print(f"Divisão da amostra entre os estratos (população finita): {tamanho_amostra_proporcao_finita}")

Tamanho da amostra MÉDIA (população finita): 1339
Divisão da amostra entre os estratos (população finita): {0: 971, 1: 321}


In [28]:
erro = 1.0  # Erro máximo de 1 unidade na média
proporcoes = {0: 0.5, 1: 0.5}

# Para uma população infinita
tamanho_amostra_media_infinita = tamanho_amostra_media_estratificada(df, 'Classe', 'Value', erro)
tamanho_amostra_proporcao_infinita = tamanho_amostra_proporcao_estratificada(df, proporcoes, tamanho_amostra_media_infinita)

print(f"Tamanho da amostra MÉDIA (população infinita): {tamanho_amostra_media_infinita}")
print(f"Divisão da amostra entre os estratos (população infinita): {tamanho_amostra_proporcao_infinita}")

Tamanho da amostra MÉDIA (população infinita): 1407
Divisão da amostra entre os estratos (população infinita): {0: 1058, 1: 349}


#### **Tamanho da amostra por conglomerados**

Aqui a unidade sorteada é o conglomerado. Em `1 estágio` calculamos quantos conglomerados $m$ sortear, usando a variância entre as médias dos conglomerados $S_c^2$. Em `2 estágios` calculamos também quantos elementos sortear dentro de cada conglomerado, usando a variância média dentro dos conglomerados $S_w^2$.

$m = \frac{M\cdot z^2_{\alpha/2}\cdot S_c^2}{(M-1)\cdot B^2 + z^2_{\alpha/2}\cdot S_c^2}$ (população finita de $M$ conglomerados)

`OBS:` *Essas expressões são uma versão simplificada. O livro apresenta as fórmulas completas, que combinam as duas variâncias.*

##### *1 Estágio*

In [29]:
def tamanho_amostra_1_estagio(df, erro_amostral, coluna, coluna_conglomerado, confianca=0.95, finita=False):

    z = norm.ppf(1 - (1 - confianca) / 2)  # Valor crítico para o nível de confiança
    medias = df.groupby(coluna_conglomerado)[coluna].mean()
    M = len(medias)  # Número de conglomerados na população
    variancia_entre = medias.var()  # Variância entre as médias dos conglomerados

    if finita:
        m = (M * (z**2) * variancia_entre) / ((M - 1) * (erro_amostral**2) + (z**2) * variancia_entre)
    else:
        m = ((z**2) * variancia_entre) / (erro_amostral**2)

    return math.ceil(m)

In [30]:
# Parâmetros para cálculo do tamanho ideal da amostra
erro_amostral = 0.5

# Número de conglomerados a sortear (método de 1 estágio)
tamanho_amostra_1_infinito = tamanho_amostra_1_estagio(df, erro_amostral, 'Value', 'Área')
tamanho_amostra_1_finito = tamanho_amostra_1_estagio(df, erro_amostral, 'Value', 'Área', finita=True)

print(f"Conglomerados a sortear (1 estágio, população infinita): {tamanho_amostra_1_infinito}")
print(f"Conglomerados a sortear (1 estágio, população finita): {tamanho_amostra_1_finito}")

Conglomerados a sortear (1 estágio, população infinita): 1
Conglomerados a sortear (1 estágio, população finita): 1


##### *2 Estágio*

In [31]:
def tamanho_amostra_2_estagios(df, erro_amostral, coluna, coluna_conglomerado, confianca=0.95, finita=False):

    z = norm.ppf(1 - (1 - confianca) / 2)  # Valor crítico para o nível de confiança

    # 1º estágio: número de conglomerados
    m = tamanho_amostra_1_estagio(df, erro_amostral, coluna, coluna_conglomerado, confianca, finita)

    # 2º estágio: elementos dentro de cada conglomerado
    variancia_dentro = df.groupby(coluna_conglomerado)[coluna].var().mean()  # Variância média dentro dos conglomerados
    tamanho_medio = df.groupby(coluna_conglomerado).size().mean()  # Tamanho médio dos conglomerados

    if finita:
        n = (tamanho_medio * (z**2) * variancia_dentro) / ((tamanho_medio - 1) * (erro_amostral**2) + (z**2) * variancia_dentro)
    else:
        n = ((z**2) * variancia_dentro) / (erro_amostral**2)

    return m, math.ceil(n)

In [32]:
# Parâmetros para cálculo do tamanho ideal da amostra
erro_amostral = 0.5

# Conglomerados e elementos por conglomerado (método de 2 estágios)
m_inf, n_inf = tamanho_amostra_2_estagios(df, erro_amostral, 'Value', 'Área')
m_fin, n_fin = tamanho_amostra_2_estagios(df, erro_amostral, 'Value', 'Área', finita=True)

print(f"2 estágios, população infinita: {m_inf} conglomerado(s) com {n_inf} elementos cada")
print(f"2 estágios, população finita: {m_fin} conglomerado(s) com {n_fin} elementos cada")

2 estágios, população infinita: 1 conglomerado(s) com 2815 elementos cada
2 estágios, população finita: 1 conglomerado(s) com 2291 elementos cada


### **Exercício 11**

Calcular o tamanho da amostra de cada região, com nível de confiança de 99% e erro amostral de 0.6, conhecendo o número de habitantes e a variância de cada uma.

In [33]:
def tamanho_amostra_estratificada(n_habitantes, variancia, erro_amostral, confianca=0.99):
    z = norm.ppf(1 - (1 - confianca) / 2)  # Valor crítico (2.576 para 99%)
    tamanho_amostra = ((z**2) * variancia * n_habitantes) / ((erro_amostral**2) * (n_habitantes - 1) + (z**2) * variancia)
    return math.ceil(tamanho_amostra)

In [34]:
# Dados das regiões
dados = {'Região': [], 'Habitantes': [], 'Variância': []}

regioes = ['Norte', 'Sul', 'Centro', 'Leste', 'Oeste']
regioesid = [1, 2, 3, 4, 5]
habitantes = [14060, 19477, 36564, 26424, 23475]
variancias = [44.5, 59.3, 82.4, 66.2, 69.5]

dados["Região"] = regioes
dados["RegiãoID"] = regioesid
dados["Habitantes"] = habitantes
dados["Variância"] = variancias

df = pd.DataFrame(dados)
df

,Região,Habitantes,Variância,RegiãoID
0,Norte,14060,44.5,1
1,Sul,19477,59.3,2
2,Centro,36564,82.4,3
3,Leste,26424,66.2,4
4,Oeste,23475,69.5,5


In [35]:
# Parâmetros para cálculo do tamanho da amostra
erro_amostral = 0.6
confianca = 0.99

# Cálculo do tamanho da amostra para cada região
tamanhos_amostra = []
for _, row in df.iterrows():
    tamanho_amostra = tamanho_amostra_estratificada(row['Habitantes'], row['Variância'], erro_amostral, confianca)
    tamanhos_amostra.append(tamanho_amostra)

# Exibição dos resultados
for i in range(len(df)):
    print(f"Tamanho da amostra para a região {df['Região'][i]}: {tamanhos_amostra[i]}")

Tamanho da amostra para a região Norte: 775
Tamanho da amostra para a região Sul: 1035
Tamanho da amostra para a região Centro: 1459
Tamanho da amostra para a região Leste: 1167
Tamanho da amostra para a região Oeste: 1215
